In [ ]:
import math
from contextlib import contextmanager

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.optim import AdamW

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class FlowerDataset(Dataset):
    def __init__(self, seqs, seq_len=20):
        self.seqs = seqs
        self.seq_len = seq_len

    def __len__(self):
        return len(self.seqs)
    
    def __getitem__(self, idx):
        x = torch.tensor(self.seqs[idx], dtype=torch.long)
        input = x[:-1]
        target = x[1:]
        return input, target


def flower_process_generation(num_samples=1000, seq_len=20, pre_depth=10, n=4, m=2, dice_probs=None):
    """
    Generate sequences from the n-m flower process.
    
    The process alternates between:
    1. Randomly selecting a die i ∈ {0, ..., n-1}, recording x_t = i
    2. Rolling that die to get outcome j ∈ {0, ..., m-1}, recording x_{t+1} = j + n
    
    Total vocabulary size: n + m tokens
    - Tokens 0 to n-1: die selection (which die was chosen)
    - Tokens n to n+m-1: die outcome (result of rolling the selected die)
    
    Args:
        num_samples: number of sequences to generate
        seq_len: number of complete cycles (each cycle = select die + roll die = 2 observations)
        pre_depth: number of initial cycles to discard (for equilibrium, though less relevant here)
        n: number of dice
        m: number of sides on each die
        dice_probs: n x m array where dice_probs[i,j] = P(die i lands on side j)
                   If None, random biased dice are generated
    
    Returns:
        data: list of sequences (each sequence has length 2*(seq_len + pre_depth))
        states: list of (die_choice, outcome) tuples for each timestep
    """
    data = []
    states = []
    
    # Generate random biased dice if not provided
    if dice_probs is None:
        # Each die gets a random bias using Dirichlet distribution
        dice_probs = np.random.dirichlet(np.ones(m), size=n)
    
    T = seq_len + pre_depth  # Total number of cycles
    
    for _ in range(num_samples):
        seq = []
        state_seq = []
        
        for t in range(T):
            # Step 1: Randomly select a die (uniform distribution)
            die_idx = np.random.randint(0, n)
            obs_die_choice = die_idx  # Observation in {0, ..., n-1}
            seq.append(obs_die_choice)
            state_seq.append(('select', die_idx))
            
            # Step 2: Roll the selected die
            die_outcome = np.random.choice(m, p=dice_probs[die_idx])
            obs_die_outcome = n + die_outcome  # Observation in {n, ..., n+m-1}
            seq.append(obs_die_outcome)
            state_seq.append(('roll', die_outcome))
        
        # Discard pre_depth cycles (2*pre_depth observations)
        seq = seq[2*pre_depth:]
        state_seq = state_seq[2*pre_depth:]
        
        data.append(seq)
        states.append(state_seq)
    
    return data, states


def Rev_flower_process_generation(num_samples=1000, seq_len=20, pre_depth=10, n=4, m=2, dice_probs=None):
    """
    Generate reversed flower process sequences.
    """
    data, states = flower_process_generation(num_samples, seq_len, pre_depth, n, m, dice_probs)
    rev_data = [list(reversed(seq)) for seq in data]
    rev_states = [list(reversed(state_seq)) for state_seq in states]
    return rev_data, rev_states


def make_flower_loader(n=4, m=2, dice_probs=None, batch_size=32, seq_len=100, 
                       num_samples=2000, shuffle=True, mode="forward"):
    """
    Create a DataLoader for the flower process.
    
    Args:
        n: number of dice
        m: number of sides per die
        dice_probs: optional n x m array of die biases
        batch_size: batch size
        seq_len: number of complete cycles (total observations = 2*seq_len)
        num_samples: number of sequences
        shuffle: whether to shuffle data
        mode: "forward" or "backward" (reversed sequences)
    
    Returns:
        DataLoader object
    """
    if mode == "backward":
        seqs, _ = Rev_flower_process_generation(
            num_samples=num_samples, 
            seq_len=seq_len, 
            n=n, 
            m=m, 
            dice_probs=dice_probs
        )
    else:
        seqs, _ = flower_process_generation(
            num_samples=num_samples, 
            seq_len=seq_len, 
            n=n, 
            m=m, 
            dice_probs=dice_probs
        )
    
    ds = FlowerDataset(seqs, seq_len=2*seq_len)  # 2*seq_len observations total
    dl = DataLoader(ds, batch_size=batch_size, shuffle=shuffle)
    return dl


# Example usage
if __name__ == "__main__":
    # Generate some example sequences
    n, m = 4, 2  # 4 dice, 2 sides each
    
    # Create specific biased dice (optional)
    # Example: first die favors side 0, second die is fair, etc.
    dice_probs = np.array([
        [0.7, 0.3],  # die 0: 70% chance of side 0
        [0.5, 0.5],  # die 1: fair
        [0.3, 0.7],  # die 2: 70% chance of side 1
        [0.6, 0.4],  # die 3: 60% chance of side 0
    ])
    
    # Generate sequences
    data, states = flower_process_generation(
        num_samples=5, 
        seq_len=10, 
        n=n, 
        m=m, 
        dice_probs=dice_probs
    )
    
    print("Example sequences (observations):")
    print(f"Vocabulary: 0-{n-1} = die selection, {n}-{n+m-1} = die outcomes")
    for i, seq in enumerate(data[:3]):
        print(f"Sequence {i}: {seq[:20]}...")  # First 20 observations
    
    # Create a DataLoader
    loader = make_flower_loader(
        n=n, 
        m=m, 
        dice_probs=dice_probs,
        batch_size=32,
        seq_len=50,
        num_samples=1000
    )
    
    print(f"\nDataLoader created with {len(loader.dataset)} sequences")
    print(f"Token vocabulary size: {n + m} (tokens 0 to {n+m-1})")


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class CoinDataset(Dataset):
    def __init__(self, seqs, seq_len = 20):
        self.num_token = 3  # Assuming 3 tokens for coin flip: 0, 1, 2
        self.seqs = seqs
        self.seq_len = seq_len

    def __len__(self):
        return len(self.seqs)
    
    def __getitem__(self, idx):
        x = torch.tensor(self.seqs[idx], dtype = torch.long)
        input = x[:-1]
        target = x[1:]
        return input, target
    
# IMPROVEMENT_PLAN.md C4.  `FlowerDataset` and `flower_process_generation`
# used to be defined HERE as well as in Flower_process_generation.py -- this
# copy with n=4/m=2 hard-coded as attributes and no `pre_depth`, that one
# parametric and with it.  Every runner imported the generator from this file
# and the Dataset from the other, so this file's FlowerDataset was dead in all
# of them.  Both copies are deleted; Flower_process_generation.py is the single
# definition.  Duplicated definitions are fine right up until one is fixed and
# the other is not.


def coin_generation(num_samples = 1000, seq_len = 20, p = 0.6, q = 0.4):
    data = []   #num_sample x seq_len
    states = [] #num_sample x seq_len
    T = seq_len

    for _ in range(num_samples):
        seq = []
        state_seq = []
        if np.random.rand() < p/(p+q):
            cur_state = 1
        else:
            if np.random.rand() < q:
                cur_state, prev_state = 0, 1
            else:
                cur_state, prev_state = 0, 0
        for t in range(T):
            state_seq.append(cur_state)
            if cur_state == 1:
                obs = 1
            elif cur_state == 0 and prev_state == 1:
                obs = 2
            else:
                obs = 0
            seq.append(obs)
            prev_state = cur_state

            if cur_state == 0:
                cur_state = 1 if np.random.rand() < p else 0
            else:
                cur_state = 0 if np.random.rand() < q else 1
        data.append(seq)
        states.append(state_seq)
    return data, states

"""
DataLoader creation function
"""

def Rev_HMM_generation(data, states):
    rev_data = [list(reversed(seq)) for seq in data]
    rev_states = [list(reversed(state_seq)) for state_seq in states]
    return rev_data, rev_states

def make_loader(data, batch_size, shuffle=True):
    """
    Plain DataLoader over `data`.

    IMPROVEMENT_PLAN.md C7.  This used to take a `states` argument that was a
    no-op: the body read `seqs, _ = data, states`, a tuple unpack that assigns
    seqs = data and discards states, written in a form that made `states` look
    load-bearing.  No caller ever used the state sequences.

    It also had a mode="backward" branch that reversed the *data* — a third,
    unused notion of "backward" alongside the triu mask and the batch swap,
    and NOT the mechanism any experiment uses.  Only pq_experiment.py calls
    this, and always with mode="forward", so the branch is removed rather than
    left as a trap.
    """
    seqs = data
    ds = CoinDataset(seqs, seq_len=len(seqs[0]))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding with optional REVERSE position assignment.

    reverse_pos=False:
        position t uses PE[t]

    reverse_pos=True:
        position t uses PE[T-1-t]   (mirror positions within current sequence length T)
    """
    def __init__(self, d_model=20, max_len=150):
        super().__init__()
        self.d_model = d_model
        self._build_pe(max_len)

    def _build_pe(self, max_len, device=None, dtype=None):
        # Ensure float dtype (sin/cos needs float)
        if dtype is None:
            dtype = torch.float32

        d_model = self.d_model
        pe = torch.zeros(max_len, d_model, device=device, dtype=dtype)

        pos = torch.arange(0, max_len, device=device, dtype=dtype).unsqueeze(1)  # (max_len, 1)
        ith = torch.arange(0, d_model, 2, device=device, dtype=dtype)            # (d_model/2,)
        div = 10000 ** (ith / d_model)                                           # (d_model/2,)

        pe[:, 0::2] = torch.sin(pos / div)
        pe[:, 1::2] = torch.cos(pos / div)

        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x, reverse_pos: bool = False):
        B, T, D = x.shape

        # Grow PE if needed (no info loss)
        if T > self.pe.size(0):
            new_len = max(T, self.pe.size(0) * 2)
            self._build_pe(new_len, device=x.device, dtype=x.dtype)

        pe_T = self.pe[:T]  # (T, D)
        if reverse_pos:
            pe_T = torch.flip(pe_T, dims=[0])  # (T, D) reversed along position axis

        return x + pe_T.unsqueeze(0)  # (B, T, D)

In [ ]:
class AttentionModel(nn.Module):
    def __init__(self, d_model=20):
        super().__init__()
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.d_model = d_model

    def forward(self, q, k, v, mask=None, return_attn=False):
        Q = self.wq(q)  # (B, T, D)
        K = self.wk(k)  # (B, T, D)
        V = self.wv(v)  # (B, T, D)

        scores = (Q @ K.transpose(-2, -1)) / (self.d_model ** 0.5)  # (B, T, T)

        if mask is not None:
            # mask True = allowed, False = blocked
            scores = scores.masked_fill(~mask, -1e9)

        prob = torch.softmax(scores, dim=-1)  # (B, T, T)
        out = prob @ V                        # (B, T, D)

        if return_attn:
            return out, prob
        return out
    

In [ ]:
def cross_ent_onehot(logits, targets):
    """
    Mean cross-entropy in BITS, and the matching perplexity 2**CE.

        -(target_prob * logits.softmax(-1).log2()).sum(dim=1)

    which returns NaN once any *non-target* class probability underflows to
    exactly 0: that term is 0 * -inf, and the NaN then propagates through the
    .sum() and poisons the whole batch even though the target class was fine.
    That is reachable here rather than hypothetical -- both processes contain
    deterministic transitions (coin token 2 -> token 1 w.p. 1; a flower roll is
    always followed by a selection), and cross-entropy training on a
    deterministic transition drives the logit gap to infinity, so a gap of ~200
    after 60-80 epochs at lr=1e-2 is realistic.  Measured: at logit scale 200
    the old expression returns nan where the true value is 0.0.

    F.cross_entropy uses the log-sum-exp trick internally, so it is exact at
    any logit scale, never touches a 0 * -inf product, and is faster (no
    separate softmax + log over (B*T, V)).  It returns nats, hence / ln 2.
    """
    C = logits.shape[-1]
    flat_input  = logits.reshape(-1, C)   # (B*T, V)
    flat_target = targets.reshape(-1)     # (B*T,)

    loss       = F.cross_entropy(flat_input, flat_target) / math.log(2)
    perplexity = 2 ** loss

    return loss, perplexity

## The causal-state bottleneck — step by step

The decoder is trained so that its prediction has to pass through a **discrete
causal state**. Each step below is one line in `OneHotDecoder.forward`.

### Step 1 — the model produces logits

The transformer stack gives a latent `x` of shape `(B, T, D)`, and
`output_prj: D → V` turns it into `logits` of shape `(B, T, V)` — one score per
vocabulary item at every position. Nothing unusual yet; these are the logits you
would normally train on directly, and they are still exposed as
`model.last_logits` so they can be plotted.

### Step 2 — argmax → one-hot: "the most prominent feature"

`states = logits.argmax(-1)` picks the winning coordinate, and `F.one_hot` turns
it into a `(B, T, V)` indicator. This is the discretisation: everything the model
knows about position `(b,t)` is compressed to *which* coordinate scored highest,
and the margin by which it won is discarded.

### Step 3 — the straight-through estimator

`argmax` has no derivative and `F.one_hot` returns a tensor that is not part of
the autograd graph, so used naively this severs the network in two. The estimator

```python
token_onehot = (hard - token_probs).detach() + token_probs
```

evaluates to `hard` in the forward pass (the `.detach()` makes the bracket a
constant) while its derivative is that of `token_probs = softmax(logits / tau)`.
Forward behaviour is exactly the hard one-hot; backward behaviour is the smooth
softmax. **This is not an optimisation detail** — without it the gradient never
reaches the transformer, which then stays at its random initialisation while the
loss still appears to fall.

`tau` is the temperature of that surrogate only; it never changes the forward
value, since `argmax` is invariant to positive scaling. Small `tau` makes the
surrogate a closer match to the hard one-hot (less bias) but saturates the
softmax (less gradient); `tau = 1` is plain softmax and is the default.

### Step 4 — the learnable causal-state matrix

`state_matrix` is a `(V, K)` parameter, where `K = causal_num` defaults to the
vocabulary size. The product

```python
causal_rep = token_onehot @ state_matrix      # (B,T,V) @ (V,K) -> (B,T,K)
```

is the **causal-state representation** of that position. Because a one-hot times
a matrix is a row lookup, this is exactly

```
causal_rep[b,t] = state_matrix[ states[b,t] ]
```

so **the causal-state label of a position is its argmax coordinate**, and there
are at most `V` distinct causal states. `K` is the *dimension* of a state vector,
not the number of states. Training learns the rows: those `V` vectors are the
"fixed description of the causal states" the model leaves behind.

### Step 5 — emission, and where the loss goes

`emission: K → V` decodes a state vector back into next-token logits, and the
cross-entropy is taken **there**. Two reasons this layer has to exist:

1. Without it the state vector would have to *be* the logits, forcing `K = V` and
   leaving the state nothing of its own to learn.
2. The loss softmaxes whatever it is handed. Handing it a one-hot would cap the
   achievable confidence at `1/(1 + (V−1)/e)` — 0.576 at `V = 3`, 0.198 at
   `V = 12` — a cross-entropy floor that *grows with the vocabulary*, i.e. an
   artefact along the very axis the flower experiments sweep.

### The logits stop being a prediction

One consequence is worth internalising before plotting anything. Nothing forces
`argmax(logits)` to be the genuinely most likely next token — the emission layer
sits downstream and can decode any code it likes, so gradient descent is free to
use the logits as a **state code** instead of a prediction.

It does exactly that. On the coin process the true predictive distributions are
`[0.4, 0.6, 0]` and `[0, 0.6, 0.4]`, which *both* peak at token 1; if the logits
were predictions the argmax would be token 1 everywhere and the bottleneck would
collapse to one state. Trained, the model instead splits them cleanly — the state
it assigns matches the true hidden state 100% of the time.

So `model.last_logits` is a `V`-way state code, and the "plot the logits directly"
view at the end shows how that code is laid out, not how the model's belief about
the next token is laid out. The predictive distribution lives in the emission
table.

### What this architecture can and cannot express

The prediction depends on the history **only through `argmax(logits)`**, so the
model can produce at most `V` distinct next-token distributions. That is a
capacity ceiling, and it is the right one to be aware of: the coin process needs
2 (it has 2 forward causal states) and has `V = 3`, so the ceiling is not
binding. A flower process with `n` dice and `m` faces needs `n + 1` and has
`V = n + m`, so it is not binding there either whenever `m ≥ 1`.

### Before using this to measure causal asymmetry — read this

The state budget is `V`, and `V` is a property of the process, not something
chosen per arm. That creates a confound specific to this study.

A flower process with `n` dice and `m` faces has `V = n + m`, needs `n + 1`
forward causal states and at most `m + 1` backward ones. So the spare capacity is

```
forward  slack = V − (n+1) = m − 1
backward slack = V − (m+1) = n − 1
```

and `C⁻ > C⁺` happens exactly when `m > n` — which is exactly when the backward
arm has *less* slack than the forward arm. Measured over a 3 × 4 grid of flower
cells, `corr(fw_slack − bw_slack, C⁻ − C⁺) = +0.95`.

**The bottleneck is tighter for one arm precisely when the hypothesis predicts
that arm should score worse.** Run as-is across the flower grid, this would
manufacture a positive ΔCE trend against `C⁻ − C⁺` with no causal asymmetry
involved — the same artefact class as a confidence penalty.

It does not affect using this notebook to *discover* states on a single process,
which is what it is for. But if the bottleneck is ever moved into the ΔCE
measurement, the state budget has to be decoupled from the vocabulary first: take
the argmax over a dedicated `D → n_states` head with `n_states` held fixed across
arms and across the grid, rather than over the `V`-dimensional logits.

### Weight decay interacts with the state vectors

`weight_decay` reaches `state_matrix` like any other parameter, and pulls all
state vectors toward the origin — measured, the per-state norms shrink about 18%
at `λ = 0.1`. The emission layer can rescale to compensate, so this need not
change the model, but it does change the *scale* of the scatter in plot 2. Keep
`λ` fixed when comparing state geometry across runs.

### One caveat the plots have to respect

`state_matrix` and `emission` compose into a single linear map, so the training
signal fixes only their **product**. Replacing `state_matrix` by `state_matrix @ Q`
and `emission` by `Q⁻¹ · emission` for any invertible `K × K` matrix `Q` leaves
every output unchanged (verified numerically: max deviation `3.6e-07`).

So the *geometry* of the state-vector scatter — distances, angles, clusters — is
**not identified by training**. It is a legitimate view of what this particular
run converged to, but it is not a property of the process. The quantity that *is*
identified is the induced predictive distribution `softmax(emission(state_matrix[v]))`
per state, which is why it gets a plot of its own.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import AdamW
from contextlib import contextmanager

import lightning as L


class OneHotDecoder(L.LightningModule):
    """
    Forward mode:
      - causal mask (tril)
      - normal positional encoding

    Backward mode:
      - anti-causal mask (triu)
      - optional reversed positional encoding

    Causal-state bottleneck (see the markdown above for the full derivation):

        x        (B,T,D)  transformer output
        logits   (B,T,V)  output_prj   D -> V
        onehot   (B,T,V)  argmax of the logits, straight-through
        rep      (B,T,K)  onehot @ state_matrix,  state_matrix is (V, K)
        out      (B,T,V)  emission     K -> V     <- the cross-entropy is here

    Because `rep` is a one-hot times a matrix, it is a ROW LOOKUP: position
    (b,t) receives row `argmax(logits[b,t])` of state_matrix.  So the state
    label of a position is its argmax token, there are at most V distinct
    causal states, and `causal_num` = K is the DIMENSION of a state vector,
    not the number of states.
    """

    def __init__(
        self,
        token_size=3,
        d_model=20,
        max_len=150,
        lr=1e-2,
        mode="forward",
        reverse_pos_for_backward: bool = False,
        n_layers=2,
        weight_decay: float = 0.0,
        causal_num: int = None,   # None -> token_size
        tau: float = 1.0,
        usage_beta: float = 0.0,
    ):
        super().__init__()

        self.mode = mode
        self.reverse_pos_for_backward = reverse_pos_for_backward
        self.n_layers = n_layers

        self.token_size = token_size
        self.d_model = d_model
        self.max_len = max_len
        self.lr = lr

        # Dimension K of a causal-state vector.  Defaults to the vocabulary
        # size.  The NUMBER of distinct states is bounded by V, not by K --
        # see the class docstring.  Read the realised count off
        # self.last_states rather than assuming either.
        self.causal_num = (
            token_size
            if causal_num is None
            else causal_num
        )

        # Softmax temperature.  tau < 1 sharpens probs toward the hard
        # one-hot, which shrinks the straight-through bias at the cost of
        # a flatter gradient.
        self.tau = tau

        # Strength of the usage-entropy penalty (see usage_penalty()).
        # 0.0 disables it, which is the collapse-prone regime.
        self.usage_beta = usage_beta

        # --------------------------------------------------
        # Training history
        # --------------------------------------------------

        self.train_loss_history = []
        self.train_perplexity_history = []
        self.usage_entropy_history = []
        self.states_occupied_history = []

        # Union of the states seen across an epoch.  Logging
        # last_states.unique().numel() per batch and letting Lightning average
        # it would give a MEAN over batches -- a fractional "number of states",
        # and an undercount, since a state used in only some batches still
        # counts as used.
        self._epoch_states = set()

        self.weight_decay = weight_decay

        # --------------------------------------------------
        # Fixed random projection
        # V -> D
        # --------------------------------------------------

        rand_prj = torch.randn(
            token_size,
            d_model
        )

        rand_prj = F.normalize(
            rand_prj,
            dim=1
        )

        self.register_buffer(
            "rand_prj",
            rand_prj
        )

        # --------------------------------------------------
        # Positional encoding
        # --------------------------------------------------

        self.pe = PositionalEncoding(
            d_model=d_model,
            max_len=max_len
        )

        # --------------------------------------------------
        # Attention layers
        # --------------------------------------------------

        self.attn_layers = nn.ModuleList([
            AttentionModel(
                d_model=d_model
            )
            for _ in range(n_layers)
        ])

        # --------------------------------------------------
        # Feed-forward layers
        # --------------------------------------------------

        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(
                    d_model,
                    4 * d_model
                ),
                nn.ReLU(),
                nn.Linear(
                    4 * d_model,
                    d_model
                ),
            )
            for _ in range(n_layers)
        ])

        # --------------------------------------------------
        # Layer normalization
        # --------------------------------------------------

        self.ln_attn = nn.ModuleList([
            nn.LayerNorm(d_model)
            for _ in range(n_layers)
        ])

        self.ln_ffn = nn.ModuleList([
            nn.LayerNorm(d_model)
            for _ in range(n_layers)
        ])

        # --------------------------------------------------
        # Output projection
        #
        # D -> V
        #
        # Here:
        # D = d_model
        # V = token_size
        # --------------------------------------------------

        # D -> V : the model's own logits.  These are the quantity the
        #          argmax is taken over, and they are exposed for plotting.
        self.output_prj = nn.Linear(
            d_model,
            token_size
        )

        # V -> K : THE LEARNABLE CAUSAL-STATE MATRIX.
        #
        #          Held as a bare Parameter rather than an nn.Linear so that
        #          `onehot @ state_matrix` is literally the operation in the
        #          specification, and so row v can be read out directly as
        #          "the causal-state vector token v is assigned to".
        self.state_matrix = nn.Parameter(
            torch.randn(token_size, self.causal_num) / (self.causal_num ** 0.5)
        )

        # K -> V : emission.  Decodes a state vector into a next-token
        #          distribution, so that K is free to differ from V and the
        #          loss stays a genuine cross-entropy.  Without this the
        #          state vector would have to BE the logits, which forces
        #          K = V and leaves nothing for the state to learn.
        self.emission = nn.Linear(
            self.causal_num,
            token_size
        )

        # --------------------------------------------------
        # Save hyperparameters
        # --------------------------------------------------

        self.save_hyperparameters()

        # --------------------------------------------------
        # Stored representations
        # --------------------------------------------------

        self.last_encodings = None

        # Filled by forward().
        #   last_logits       (B,T,V)  pre-bottleneck logits
        #   last_token_probs  (B,T,V)  softmax of those, the STE surrogate
        #   last_states       (B,T)    argmax token = the causal-state label
        #   last_causal_reps  (B,T,K)  the state vector each position got
        self.last_logits = None

        self.last_token_probs = None

        self.last_states = None

        self.last_causal_reps = None

        self.last_attention = None

        self.last_attention_layers = None

        # --------------------------------------------------
        # Attention storage
        # --------------------------------------------------

        self.store_attention = False

        # --------------------------------------------------
        # Mask cache
        # --------------------------------------------------

        self._mask_cache = {}


    # ======================================================
    # ATTENTION MASK
    # ======================================================

    def _causal_mask(
        self,
        T: int,
        device
    ):


        key = (
            T,
            str(device)
        )

        cached = self._mask_cache.get(key)

        if cached is not None:
            return cached

        ones = torch.ones(
            (T, T),
            device=device,
            dtype=torch.bool
        )

        if self.mode == "forward":

            mask = torch.tril(
                ones
            ).unsqueeze(0)

        elif self.mode == "backward":

            mask = torch.triu(
                ones
            ).unsqueeze(0)

        else:

            raise ValueError(
                f"Invalid mode: {self.mode}. "
                "Must be 'forward' or 'backward'."
            )

        self._mask_cache[key] = mask

        return mask


    # ======================================================
    # ATTENTION CAPTURE
    # ======================================================

    @contextmanager
    def capture_attention(self):

        """
        Temporarily retain attention maps.
        """

        previous = self.store_attention

        self.store_attention = True

        try:
            yield self

        finally:
            self.store_attention = previous


    # ======================================================
    # FORWARD
    # ======================================================

    def forward(
        self,
        tokens
    ):

        # --------------------------------------------------
        # 1. Sanitize token dtype
        # --------------------------------------------------

        if isinstance(tokens, torch.Tensor):

            if tokens.dtype in (
                torch.float32,
                torch.float64
            ):
                tokens = tokens.long()

            elif tokens.dtype not in (
                torch.long,
                torch.int64
            ):
                tokens = tokens.long()

        else:

            tokens = torch.LongTensor(tokens).to(
                self.rand_prj.device
            )


        # --------------------------------------------------
        # 2. Input one-hot
        #
        # tokens:
        #     (B, T)
        #
        # one_hot:
        #     (B, T, V)
        # --------------------------------------------------

        one_hot = F.one_hot(
            tokens,
            num_classes=self.token_size
        ).float()


        # --------------------------------------------------
        # 3. Fixed random projection
        #
        # (B,T,V) @ (V,D)
        #
        # -> (B,T,D)
        # --------------------------------------------------

        x = one_hot @ self.rand_prj


        # --------------------------------------------------
        # 4. Positional encoding
        # --------------------------------------------------

        reverse_pos = (
            self.mode == "backward"
            and self.reverse_pos_for_backward
        )

        x = self.pe(
            x,
            reverse_pos=reverse_pos
        )


        # --------------------------------------------------
        # 5. Attention mask
        # --------------------------------------------------

        B, T, D = x.shape
        mask = self._causal_mask(
            T,
            x.device
        )


        # --------------------------------------------------
        # 6. Transformer layers
        # --------------------------------------------------

        want_attn = self.store_attention

        attn_maps = []


        for (
            attn,
            ffn,
            ln1,
            ln2
        ) in zip(
            self.attn_layers,
            self.ffn_layers,
            self.ln_attn,
            self.ln_ffn
        ):

            # ----------------------------------------------
            # Attention
            # ----------------------------------------------

            normed = ln1(x)

            if want_attn:

                attn_out, attn_prob = attn(
                    normed,
                    normed,
                    normed,
                    mask=mask,
                    return_attn=True
                )

                attn_maps.append(
                    attn_prob.detach()
                )

            else:

                attn_out = attn(
                    normed,
                    normed,
                    normed,
                    mask=mask,
                    return_attn=False
                )

            x = x + attn_out


            normed = ln2(x)

            x = x + ffn(normed)


        self.last_encodings = x

        self.last_attention_layers = (
            attn_maps
            if want_attn
            else None
        )

        self.last_attention = (
            attn_maps[-1]
            if attn_maps
            else None
        )

        # --------------------------------------------------
        # 7. The model's logits
        #
        # x:      (B, T, D)
        # logits: (B, T, V)
        # --------------------------------------------------

        logits = self.output_prj(x)

        token_probs = F.softmax(
            logits / self.tau,
            dim=-1
        )


        # --------------------------------------------------
        # 8. Argmax -> one-hot, with a straight-through estimator
        #
        # "retrieve the most prominent feature in the encoding":
        # the winning coordinate of the logit vector.
        #
        # argmax has no gradient and F.one_hot returns a leaf, so
        # the estimator is mandatory, not a refinement -- without it
        # the graph is severed here and every layer above stays at
        # its initial values while the loss still appears to fall.
        #
        # Forward value:     hard one-hot
        # Backward gradient: through token_probs
        # --------------------------------------------------

        states = logits.argmax(
            dim=-1
        )

        hard = F.one_hot(
            states,
            num_classes=self.token_size
        ).float()

        token_onehot = (
            hard - token_probs
        ).detach() + token_probs


        # --------------------------------------------------
        # 9. One-hot -> causal-state vector
        #
        # (B,T,V) @ (V,K) -> (B,T,K)
        #
        # The one-hot selects a ROW, so this is a lookup:
        #
        #     causal_rep[b,t] = state_matrix[ states[b,t] ]
        #
        # Two consequences that the plots below depend on:
        #
        #   * causal_rep takes at most V distinct values, so the
        #     causal-state LABEL of a position is its argmax token.
        #   * the gradient still reaches state_matrix through the
        #     straight-through one-hot, so the rows are learned --
        #     these rows are the "fixed description of the causal
        #     state vectors" left behind by training.
        # --------------------------------------------------

        causal_rep = token_onehot @ self.state_matrix


        # --------------------------------------------------
        # 10. Emission: state vector -> next-token logits
        #
        # (B,T,K) @ (K,V) -> (B,T,V)
        #
        # Returns real-valued LOGITS.  The loss softmaxes whatever
        # it is given, so returning a one-hot instead would cap the
        # achievable confidence at 1/(1 + (V-1)/e) -- 0.576 at V=3,
        # 0.198 at V=12 -- a cross-entropy floor that GROWS WITH THE
        # VOCABULARY, i.e. an artefact along the very axis the
        # flower experiments sweep.
        # --------------------------------------------------

        out_logits = self.emission(causal_rep)


        self.last_logits = logits

        self.last_token_probs = token_probs

        self.last_states = states

        self.last_causal_reps = causal_rep


        return out_logits

    # ======================================================
    # USAGE-ENTROPY PENALTY
    # ======================================================

    def usage_penalty(self, probs):

        """
        Penalise collapse of the argmax onto a subset of the vocabulary.

        probs is (B,T,V), the softmax of the logits.  Let

            p_bar[v] = mean_{b,t} probs[b,t,v]

        be the state distribution MARGINALISED over the batch and over
        positions, and H(p_bar) its entropy in bits.  The penalty

            beta * (log2(V) - H(p_bar))

        is >= 0 and vanishes at uniform usage.  It is the entropy of the
        MARGINAL, not the mean of the per-position entropies: a flat marginal
        says every state gets used somewhere in the data, while each single
        position stays free to commit hard to one.  Penalising the mean
        per-position entropy instead would fight the bottleneck rather than
        spread it.

        WARNING, specific to this architecture.  The quantity being spread is
        now the model's own logit distribution, and those logits double as the
        state code, so a large beta buys state diversity by distorting the
        code.  Default is 0.0.  Use a small beta briefly to escape a collapsed
        run, then anneal it to zero and report the unpenalised model.
        """

        p_bar = probs.reshape(-1, self.token_size).mean(dim=0)

        H_usage = -(
            p_bar * torch.log2(p_bar + 1e-12)
        ).sum()

        H_max = math.log2(self.token_size)

        return H_usage, self.usage_beta * (H_max - H_usage)


    def training_step(
        self,
        batch,
        batch_idx
    ):

        inputs, targets = batch
        logits = self(inputs)

        loss, perplexity = cross_ent_onehot(
            logits,
            targets
        )

        # ------------------------------------------------------
        # The CE is logged SEPARATELY from the penalised objective.
        # Only the CE is comparable to H_inf; the penalised total is
        # not a cross-entropy and must never be quoted as one.
        # ------------------------------------------------------

        H_usage, penalty = self.usage_penalty(
            self.last_token_probs
        )

        total = loss + penalty


        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

        self.log(
            "train_perplexity",
            perplexity,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

        self.log(
            "state_usage_entropy",
            H_usage,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

        self._epoch_states.update(
            self.last_states.unique().tolist()
        )

        return total

    def on_train_epoch_end(self):

        loss = self.trainer.callback_metrics.get(
            "train_loss"
        )

        perplexity = self.trainer.callback_metrics.get(
            "train_perplexity"
        )


        if loss is not None:

            self.train_loss_history.append(
                loss.detach().cpu().item()
            )


        if perplexity is not None:

            self.train_perplexity_history.append(
                perplexity.detach().cpu().item()
            )


        usage = self.trainer.callback_metrics.get(
            "state_usage_entropy"
        )

        occupied = len(self._epoch_states)

        self._epoch_states = set()


        if usage is not None:

            self.usage_entropy_history.append(
                usage.detach().cpu().item()
            )


        self.states_occupied_history.append(occupied)


    # ======================================================
    # OPTIMIZER
    # ======================================================

    def configure_optimizers(self):

        return AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay
        )

In [ ]:
causal_num = 10
C = torch.tensor([1, 2, 4, 23, 5,3 ,23, 25 ,35  ,32 ,2,4 , 34])
one_hot_vector = F.one_hot(C.argmax(dim=-1), num_classes=causal_num)
print(one_hot_vector)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assume your high-dimensional vector has 128 features
high_dim_size = 128
batch_size = 4

# Create random input tensor of shape (batch_size, high_dim_size)
high_dim_vectors = torch.randn(batch_size, high_dim_size)

# Step 1: Compress from 128 dimensions down to 3 dimensions
projection_layer = nn.Linear(high_dim_size, 3)
logits3d = projection_layer(high_dim_vectors)  # Shape: (batch_size, 3)
print(high_dim_vectors)
print(logits3d)


# Testing new training method

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import lightning as L
import umap

# ---------------------------------------------------------
# Generate HMM / coin data
# ---------------------------------------------------------

P_COIN, Q_COIN = 0.6, 0.4

data, states = coin_generation(
    num_samples=2000,
    seq_len=100,
    p=P_COIN,
    q=Q_COIN
)

fw_loader = make_loader(
    data,
    batch_size=32,
    shuffle=True
)

print("Number of sequences:", len(data))
print("Sequence length:", len(data[0]))
print("Input vocabulary:", np.unique(data))

# Check one batch
batch_inputs, batch_targets = next(iter(fw_loader))

print("\nBatch shapes")
print("Inputs :", batch_inputs.shape)
print("Targets:", batch_targets.shape)

In [ ]:
model = OneHotDecoder(
    token_size=3,
    d_model=20,
    max_len=150,
    lr=1e-3,
    mode="forward",
    n_layers=2,
    weight_decay=0.0,
    causal_num=None,   # K, the state-vector DIMENSION.  None -> vocabulary size.
    tau=1.0,           # temperature of the straight-through surrogate only
    usage_beta=0.0,    # anti-collapse penalty; see usage_penalty() for the caveat
)

# --------------------------------------------------
# Sanity check 1 -- shapes
# --------------------------------------------------

with torch.no_grad():
    out_logits = model(batch_inputs)

print("Input        :", tuple(batch_inputs.shape))
print("logits       :", tuple(model.last_logits.shape),      "  (D -> V)")
print("state_matrix :", tuple(model.state_matrix.shape),     "  (V, K)")
print("causal_rep   :", tuple(model.last_causal_reps.shape), "  (B, T, K)")
print("out_logits   :", tuple(out_logits.shape),             "  (K -> V)")

# --------------------------------------------------
# Sanity check 2 -- causal_rep really is a row lookup
#
# one_hot @ state_matrix selects a row, so every position must
# receive exactly state_matrix[argmax], and there can be at most
# V distinct causal-state vectors no matter how large K is.
# --------------------------------------------------

flat = model.last_causal_reps.reshape(-1, model.causal_num)
print(f"\nDistinct causal_rep vectors: {torch.unique(flat, dim=0).shape[0]} "
      f"(bounded by V = {model.token_size})")

t0 = model.last_states[0, 0]
print("causal_rep[0,0] == state_matrix[argmax] :",
      torch.allclose(model.last_causal_reps[0, 0], model.state_matrix[t0]))

# --------------------------------------------------
# Sanity check 3 -- the straight-through estimator passes gradient
#
# This is not optional.  An earlier version of this model took the
# argmax without the estimator, which cut the graph: 2 of 33 parameter
# tensors trained and the transformer stayed at random initialisation,
# while the loss curve still looked plausible.  A silent failure like
# that needs an explicit check.
# --------------------------------------------------

loss, _ = cross_ent_onehot(model(batch_inputs), batch_targets)
loss.backward()

live = [n for n, p in model.named_parameters()
        if p.grad is not None and p.grad.abs().sum() > 0]
dead = [n for n, p in model.named_parameters()
        if p.grad is None or p.grad.abs().sum() == 0]

print(f"\nParameters receiving gradient: {len(live)}/{len(live) + len(dead)}")
if dead:
    print("NO gradient:", dead)
else:
    print("  state_matrix and emission included:",
          "state_matrix" in live and any("emission" in n for n in live))

model.zero_grad()

In [ ]:
trainer = L.Trainer(
    max_epochs=50,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
)

trainer.fit(
    model,
    train_dataloaders=fw_loader
)

In [ ]:
plt.figure(figsize=(7, 5))

epochs = np.arange(
    1,
    len(model.train_loss_history) + 1
)

plt.plot(
    epochs,
    model.train_loss_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss (bits)")
plt.title("Training Loss")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    epochs,
    model.train_perplexity_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.title("Training Perplexity")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------
# Collapse diagnostic.
#
# states_occupied is the count that matters: the number of causal
# states the model ACTUALLY uses over the epoch.  It is bounded by the
# VOCABULARY SIZE, because the state label is the argmax coordinate --
# causal_num is the dimension of a state vector, not a bound on how
# many states there are.
#
# If it falls to 1 the bottleneck has collapsed and the model is
# emitting the marginal distribution regardless of context.  The loss
# curve alone will not tell you this.
#
# Usage entropy is the softer reading of the same thing: log2(V) means
# every coordinate wins the argmax equally often, 0 means total
# collapse.  Uniform is NOT the target -- the coin process has 2
# forward causal states, so the right answer here is near 1 bit, not
# log2(3) = 1.585.
# --------------------------------------------------

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

epochs = np.arange(1, len(model.states_occupied_history) + 1)

ax[0].plot(epochs, model.states_occupied_history, marker="o", color="#3B6EA5")
ax[0].axhline(model.token_size, ls="--", c="#9AA5B1",
              label=f"vocabulary size = {model.token_size}")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("States occupied")
ax[0].set_title("Causal states actually used")
ax[0].set_ylim(0, model.token_size + 0.4)
ax[0].legend(frameon=False)

ax[1].plot(epochs, model.usage_entropy_history, marker="o", color="#3B6EA5")
ax[1].axhline(math.log2(model.token_size), ls="--", c="#9AA5B1",
              label="log2(V) = uniform")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Usage entropy (bits)")
ax[1].set_title("Argmax usage entropy")
ax[1].legend(frameon=False)

for a in ax:
    a.spines[["top", "right"]].set_visible(False)
    a.grid(alpha=0.25, lw=0.6)
    a.set_axisbelow(True)

plt.tight_layout()
plt.show()

## Reading the causal states off a trained model

Three views, in increasing order of how much they can be trusted.

| plot | what it shows | identified? |
|---|---|---|
| occupancy | how many token positions were assigned to each causal state | yes |
| state-vector scatter | where the `V` rows of `state_matrix` sit relative to each other | **no** — only up to an invertible `K × K` map |
| emission heatmap | `P(next token \| state)` for each occupied state | yes |

The occupancy plot is the diagnostic that matters most day to day: if it puts
everything in one bar the bottleneck has collapsed and the model is emitting the
marginal distribution regardless of context, which the loss curve alone will not
tell you.

In [ ]:
def causal_state_report(model, data_loader, min_pos=0):

    """
    Assign every token position to a causal state and summarise the result.

    min_pos skips the first few positions of each sequence.  A forward model at
    position t has only t tokens of context, so the earliest positions are not
    predicting from a settled state; set min_pos > 0 to exclude them.

    Returns a dict with
        counts     (V,)    token positions assigned to each state
        vectors    (V,K)   the learned state vectors, i.e. state_matrix
        emissions  (V,V)   P(next token | state), the identified quantity
        occupied   list    state ids with a non-zero count
    """

    was_training = model.training

    model.eval()

    device = next(model.parameters()).device

    counts = torch.zeros(
        model.token_size,
        dtype=torch.long
    )

    with torch.no_grad():

        for inputs, _ in data_loader:

            model(inputs.to(device))

            states = model.last_states[:, min_pos:].reshape(-1)

            counts += torch.bincount(
                states.cpu(),
                minlength=model.token_size
            )

        vectors = model.state_matrix.detach().cpu()

        emissions = model.emission(
            model.state_matrix
        ).softmax(dim=-1).detach().cpu()

    if was_training:
        model.train()

    return {
        "counts": counts,
        "vectors": vectors,
        "emissions": emissions,
        "occupied": [
            v for v in range(model.token_size)
            if counts[v] > 0
        ],
    }


report = causal_state_report(model, fw_loader, min_pos=5)

counts = report["counts"].numpy()

print("causal states available (= vocabulary size):", model.token_size)
print("causal states occupied                     :", report["occupied"])
print("state vector dimension K                   :", model.causal_num)
print()
for v in range(model.token_size):
    n = int(counts[v])
    tag = "" if n else "   (unoccupied)"
    print(f"  state {v}: {n:>8,d} token positions{tag}")

In [ ]:
# ==========================================================
# PLOT 1 -- occupancy: how many tokens lie in each causal state
# ==========================================================

ids = np.arange(model.token_size)

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.bar(
    ids,
    counts,
    color=["#3B6EA5" if c > 0 else "#D8DEE6" for c in counts],
    width=0.62,
)

for v, c in zip(ids, counts):
    ax.text(
        v,
        c + counts.max() * 0.02,
        f"{c:,}",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#3C4653",
    )

ax.set_xlabel("Causal state  (= argmax coordinate)")
ax.set_ylabel("Token positions assigned")
ax.set_title("Causal-state occupancy")
ax.set_xticks(ids)
ax.set_ylim(0, counts.max() * 1.12)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25, lw=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

if len(report["occupied"]) == 1:
    print("COLLAPSED: every position is in one state -- the model is emitting "
          "the marginal distribution regardless of context.")
else:
    print(f"{len(report['occupied'])} of {model.token_size} available states are in use.")

In [ ]:
# ==========================================================
# PLOT 2 -- where the causal states sit relative to each other
#
# One point per causal state, labelled with its token count.
# K may exceed 2, so the vectors are projected with PCA.
#
# NOTE the geometry here is NOT identified by training (see the
# markdown above): state_matrix @ Q with emission @ inv(Q) gives an
# identical model for any invertible Q.  Read this as "what this run
# converged to", not as a property of the process.  Plot 3 is the
# view that does not have this problem.
# ==========================================================

from sklearn.decomposition import PCA

vectors = report["vectors"].numpy()

if min(model.causal_num, model.token_size) >= 2:
    coords = PCA(n_components=2, random_state=0).fit_transform(vectors)
    axis_label = "PCA component"
else:
    coords = np.column_stack([vectors[:, 0], np.zeros(model.token_size)])
    axis_label = "state vector"

fig, ax = plt.subplots(figsize=(7.5, 6))

live = counts > 0

ax.scatter(
    coords[~live, 0],
    coords[~live, 1],
    s=90,
    facecolor="none",
    edgecolor="#B4BCC7",
    linewidth=1.6,
    zorder=2,
)

sc = ax.scatter(
    coords[live, 0],
    coords[live, 1],
    s=180 + 900 * counts[live] / counts.max(),
    c=counts[live],
    cmap="Blues",
    vmin=0,
    edgecolor="white",
    linewidth=2.0,
    zorder=3,
)

for v in range(model.token_size):
    label = f"state {v}\n{counts[v]:,} tokens" if counts[v] else f"state {v}\nunused"
    ax.annotate(
        label,
        (coords[v, 0], coords[v, 1]),
        textcoords="offset points",
        xytext=(0, -34),
        ha="center",
        fontsize=9,
        color="#3C4653",
    )

cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Token positions assigned")
cbar.outline.set_visible(False)

ax.set_xlabel(f"{axis_label} 1")
ax.set_ylabel(f"{axis_label} 2")
ax.set_title("Learned causal-state vectors\n"
             "(hollow = state exists in state_matrix but no token uses it)")

ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.25, lw=0.6)
ax.set_axisbelow(True)
ax.margins(0.22)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# PLOT 3 -- what each causal state actually predicts
#
# P(next token | state) = softmax(emission(state_matrix[v])).
# This is the identified quantity: unlike the raw state vectors it
# is invariant to the K x K reparameterisation, so it is the one to
# compare against the epsilon-machine's emission probabilities.
# ==========================================================

emissions = report["emissions"].numpy()
occ = report["occupied"]

fig, ax = plt.subplots(figsize=(1.6 + 1.1 * model.token_size, 1.1 + 0.7 * len(occ)))

im = ax.imshow(
    emissions[occ],
    cmap="Blues",
    vmin=0,
    vmax=1,
    aspect="auto",
)

for r, v in enumerate(occ):
    for t in range(model.token_size):
        val = emissions[v, t]
        ax.text(
            t, r, f"{val:.3f}",
            ha="center", va="center", fontsize=9,
            color="white" if val > 0.55 else "#3C4653",
        )

ax.set_xticks(range(model.token_size))
ax.set_yticks(range(len(occ)))
ax.set_yticklabels([f"state {v}  (n={counts[v]:,})" for v in occ])
ax.set_xlabel("Next token")
ax.set_title("P(next token | causal state)")

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Probability")
cbar.outline.set_visible(False)

ax.set_xticks(np.arange(-.5, model.token_size, 1), minor=True)
ax.set_yticks(np.arange(-.5, len(occ), 1), minor=True)
ax.grid(which="minor", color="white", lw=2)
ax.tick_params(which="minor", length=0)

plt.tight_layout()
plt.show()

print("Two states with (nearly) identical rows are the same causal state "
      "reached by two different argmax coordinates.")

In [ ]:
# ==========================================================
# Does it recover the ACTUAL causal states?
#
# The coin process is small enough to check against the closed form.
# Hidden state 0 emits token 0 (if it stayed) or token 2 (if it just
# arrived), hidden state 1 emits token 1, so
#
#     P(next | hidden 0) = [1-p,  p,   0 ]
#     P(next | hidden 1) = [ 0,  1-q,  q ]
#
# coin_generation returns the true hidden states, so the model's
# assignment can be scored directly rather than eyeballed.
# ==========================================================

emissions = report["emissions"].numpy()

theory = {
    0: np.array([1 - P_COIN, P_COIN, 0.0]),
    1: np.array([0.0, 1 - Q_COIN, Q_COIN]),
}

MIN_POS = 5

true_states = torch.tensor(np.array(states))[:, :-1]
inputs_all = torch.tensor(np.array(data))[:, :-1]

model.eval()
assigned = []
with torch.no_grad():
    for i in range(0, len(inputs_all), 256):
        model(inputs_all[i:i + 256])
        assigned.append(model.last_states)
assigned = torch.cat(assigned)

a = assigned[:, MIN_POS:].reshape(-1).numpy()
h = true_states[:, MIN_POS:].reshape(-1).numpy()

print("model state  vs  true hidden state")
for ms in sorted(set(a.tolist())):
    sel = h[a == ms]
    print(f"  state {ms}: hidden0 {100 * (sel == 0).mean():5.1f}%   "
          f"hidden1 {100 * (sel == 1).mean():5.1f}%   (n = {sel.size:,})")

print("\nlearned emission  vs  closed form")
for ms in sorted(set(a.tolist())):
    dom = int(np.bincount(h[a == ms], minlength=2).argmax())
    dev = np.abs(emissions[ms] - theory[dom]).max()
    print(f"  state {ms} ~ hidden {dom}")
    print(f"     learned = {np.round(emissions[ms], 3)}")
    print(f"     theory  = {np.round(theory[dom], 3)}     max deviation {dev:.4f}")

print(f"\nStates recovered: {len(set(a.tolist()))}   "
      f"(C+ = 2 forward causal states for the coin)")

In [ ]:
def extract_final_latents(model, data_loader):

    model.eval()

    device = next(model.parameters()).device

    all_latents = []
    all_tokens = []
    all_targets = []

    with torch.no_grad():

        for inputs, targets in data_loader:

            inputs = inputs.to(device)
            targets = targets.to(device)

            logits = model(inputs)

            # (B, T, d_model)
            latent = model.last_encodings

            # Final position of each sequence
            final_latent = latent[:, -1, :]

            # Token that produced this final latent
            final_token = inputs[:, -1]

            all_latents.append(
                final_latent.cpu()
            )

            all_tokens.append(
                final_token.cpu()
            )

            all_targets.append(
                targets[:, -1].cpu()
            )

    return (
        torch.cat(all_latents, dim=0),
        torch.cat(all_tokens, dim=0),
        torch.cat(all_targets, dim=0)
    )

def extract_final_logits(model, data_loader):
    model.eval()
    device = next(model.parameters()).device

    all_logits = []
    all_tokens = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in data_loader:

            inputs = inputs.to(device)
            targets = targets.to(device) 

            model(inputs)

            # "the logit" in the specification is the model's OWN logits,
            # the quantity the argmax is taken over -- not the post-emission
            # output.  Use model.last_logits for that; the return value of
            # forward() is the emission output the loss is computed on.
            logits = model.last_logits

            final_logit = logits[:, -1, :]
            final_token = inputs[:, -1]
            final_target = targets[:, -1]

            all_logits.append(final_logit.cpu())
            all_tokens.append(final_token.cpu())
            all_targets.append(final_target.cpu())

    return (
        torch.cat(all_logits, dim = 0),
        torch.cat(all_tokens, dim=0),
        torch.cat(all_targets, dim=0)
    )

In [ ]:
latents, tokens, targets = extract_final_latents(
    model,
    fw_loader
)

logits, tokens, targets = extract_final_logits(
    model,
    fw_loader
)

print("Logits:", logits.shape)
print("Latents:", latents.shape)
print("Tokens :", tokens.shape)
print("Targets:", targets.shape)

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    random_state=42
)

embedding = reducer.fit_transform(
    logits.numpy()
)

print(embedding.shape)

In [ ]:
plt.figure(figsize=(8, 6))

for token in torch.unique(tokens):

    mask = tokens == token

    plt.scatter(
        embedding[mask, 0],
        embedding[mask, 1],
        s=15,
        alpha=0.7,
        label=f"Token {token.item()}"
    )

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("Transformer Latent Space Colored by Token")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()